# Hyperparameter Fine-Tuning

We have decided to use `XGBoost` as our final model 
In this notebook, we will: 
- Define important `XGBoost` hyperparameters 
- Conduct hyperparameter fine-tuning using `Optuna` on a wide search space to narrow down key values

In [3]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import joblib
import optuna
import os
import pandas as pd
import seaborn as sns
from collections import Counter
from sklearn._config import set_config
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    confusion_matrix,
    fbeta_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from xgboost import XGBClassifier

## Important XGBoost parameters
Link to documentation: https://xgboost.readthedocs.io/en/stable/python/python_api.html#module-xgboost.sklearn

Blah blah blah blah (refer to documentation for this)

## Hyperparameter Fine-Tuning Using Optuna

### Get Cross Validation Splits

In [2]:
filepath_xtrain = "/Users/thananpornsethjinda/Desktop/credit-risk-modeling/data/interim/xtrain.csv" # filepath to train split 

filepath_ytrain = "/Users/thananpornsethjinda/Desktop/credit-risk-modeling/data/interim/ytrain.csv" # filepath to train split 

X_train = pd.read_csv(filepath_xtrain)

y_train = pd.read_csv(filepath_ytrain)

In [3]:

splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

strat_splits = []

for train_index, val_index in splitter.split(X_train, y_train): 

    strat_train_set_n = X_train.iloc[train_index]

    strat_y_train_set_n = y_train.iloc[train_index]['loan_status']

    strat_val_set_n = X_train.iloc[val_index]

    strat_y_val_set_n = y_train.iloc[val_index]['loan_status']

    strat_splits.append([strat_train_set_n, strat_val_set_n, strat_y_train_set_n, strat_y_val_set_n])

### Final Preprocessing Steps

In [4]:
FEATURES = ['int_rate','fico_range_high','inq_last_6mths','open_il_12m','acc_open_past_24mths','mort_acc','num_tl_op_past_12m','percent_bc_gt_75',
    'term','sub_grade']

NUMERICAL_FEATURES = ['int_rate','fico_range_high','inq_last_6mths','open_il_12m','acc_open_past_24mths','mort_acc','num_tl_op_past_12m','percent_bc_gt_75']

CATEGORICAL_FEATURES = ['term', 'sub_grade']

impute = ColumnTransformer(
    [
        ("numerical_imputation", SimpleImputer(strategy='median'), NUMERICAL_FEATURES), 
        ("categorical_impuutation", SimpleImputer(strategy='most_frequent'), CATEGORICAL_FEATURES)
    ], 
    remainder='passthrough',  
    verbose_feature_names_out=False
)

encode = ColumnTransformer(
    [
        ("one_hot_encoding", OneHotEncoder(drop='first', sparse_output=False, dtype=np.int64), ['term']), 
        ("ordinal_encoding", OrdinalEncoder(dtype=np.int64), ['sub_grade']), 
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

scale = make_column_transformer(
    (RobustScaler(), NUMERICAL_FEATURES),
    remainder='passthrough', 
    verbose_feature_names_out=False
)

In [5]:
final_pipeline = Pipeline([
            ("imputation",impute),
            ("encoding",encode),
            ("feature_scaling", scale),
            ("xgboost_model", XGBClassifier())
        ])

### Setting Up and Tuning

In [6]:
%env MLFLOW_TRACKING_URI = sqlite:///../mlflow.db 

env: MLFLOW_TRACKING_URI=sqlite:///../mlflow.db


In [7]:
ARTIFACT_PATH = "/Users/thananpornsethjinda/Desktop/credit-risk-modeling/mlruns"
EXPERIMENT_NAME = "hyper-parameter-fine-tuning"
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=ARTIFACT_PATH)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='/Users/thananpornsethjinda/Desktop/credit-risk-modeling/mlruns', creation_time=1775357138678, experiment_id='6', last_update_time=1775357138678, lifecycle_stage='active', name='hyper-parameter-fine-tuning', tags={}, workspace='default'>

In [8]:
# override Optuna's default logging to ERROR only
optuna.logging.set_verbosity(optuna.logging.ERROR)

# define a logging callback that will report on only new challenger parameter configurations if a
# trial has usurped the state of 'best conditions'

def champion_callback(study, frozen_trial):
  """
  Logging callback that will report when a new trial iteration improves upon existing
  best trial values.

  Note: This callback is not intended for use in distributed computing systems such as Spark
  or Ray due to the micro-batch iterative implementation for distributing trials to a cluster's
  workers or agents.
  The race conditions with file system state management for distributed trials will render
  inconsistent values with this callback.
  """

  winner = study.user_attrs.get("winner", None)

  if study.best_value and winner != study.best_value:
      study.set_user_attr("winner", study.best_value)
      if winner:
          improvement_percent = (abs(winner - study.best_value) / study.best_value) * 100
          print(
              f"Trial {frozen_trial.number} achieved value: {frozen_trial.value} with "
              f"{improvement_percent: .4f}% improvement"
          )
      else:
          print(f"Initial trial {frozen_trial.number} achieved value: {frozen_trial.value}")

In [ ]:
def train(pipeline, experiment_name): 

    y_pred_val = []

    avg_train_recall = []
    avg_val_recall = []
    avg_train_precision = []
    avg_val_precision = []
    avg_train_fbeta = []
    avg_val_fbeta = []

    model_name = list(pipeline.named_steps)[-1]

    mlflow.set_tags({
        "model_type": model_name,
        "experiment_phase": experiment_name,
    }) # set tags 

    for index in range(len(strat_splits)): 

        Xtrain, Xval, ytrain, yval = strat_splits[index]

        Xtrain, Xval, ytrain, yval = Xtrain[FEATURES], Xval[FEATURES], ytrain.map({'Charged Off':1, 'Fully Paid':0}), yval.map({'Charged Off':1, 'Fully Paid':0})

        pipeline.fit(Xtrain, ytrain)

        # get the metrics (recall, precision, F2)

        y_train_pred = pipeline.predict(Xtrain)
        y_val_pred = pipeline.predict(Xval)

        y_pred_val.append(y_val_pred)

        # recall, precision # may encounter issues if they are not numerical so if you do convert them to nbinary 0s and 1s 

        recall_train = recall_score(ytrain, y_train_pred)
        avg_train_recall.append(recall_train)

        recall_val = recall_score(yval, y_val_pred)
        avg_val_recall.append(recall_val)

        precision_train = precision_score(ytrain, y_train_pred)
        avg_train_precision.append(precision_train)

        precision_val = precision_score(yval, y_val_pred)
        avg_val_precision.append(precision_val)

        fbeta_train = fbeta_score(ytrain, y_train_pred, beta=2)
        avg_train_fbeta.append(fbeta_train)

        fbeta_val = fbeta_score(yval, y_val_pred, beta=2)
        avg_val_fbeta.append(fbeta_val)

    parent_metrics = {
                    "train_recall": np.mean(avg_train_recall),
                    "validation_recall": np.mean(avg_val_recall), 
                    "train_precision": np.mean(avg_train_precision),
                    "validation_precision": np.mean(avg_val_precision),
                    "train_fbeta": np.mean(avg_train_fbeta),
                    "validation_fbeta": np.mean(avg_val_fbeta)
                }

    mlflow.log_metrics(metrics=parent_metrics)

    return parent_metrics["validation_recall"]

In [ ]:
# define optuna function with mlflow 

y = strat_splits[0][2]
y_numeric = y.map({'Charged Off':1, 'Fully Paid':0})

counter = Counter(y_numeric)
class_ratio = counter[0]/counter[1]

def objective(trial): 
        
    with mlflow.start_run(nested=True):

        # define parameters 

        params = {
            'random_state': 42, 
            'scale_pos_weight': class_ratio,
            'verbosity': 0, 
            'n_estimators': trial.suggest_int('n_estimators',  50, 1000), 
            'booster': trial.suggest_categorical('booster', ['gbtree']),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
            'max_depth': trial.suggest_int('max_depth', 1, 7),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 7), 
            'objective': trial.suggest_categorical('objective', ['binary:logistic', 'binary:logitraw', 'binary:hinge'])
        }

        mlflow.log_params(params=params)

        model = XGBClassifier(**params)

        training_pipeline = Pipeline([
            ("imputation",impute),
            ("encoding",encode),
            ("feature_scaling", scale),
            ("xgboost_model", model)
        ])

        ## so for each hyperparameter combination we will do cross validation 

        recall = train(training_pipeline, EXPERIMENT_NAME)

        return recall

In [24]:
run_name="XGBoost"
set_config(transform_output="pandas")
with mlflow.start_run(run_name=run_name): 

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=200, callbacks=[champion_callback])
    print("Optimisation works!")

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_test_recall", study.best_value)

/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and 

Initial trial 1 achieved value: 0.4176799782757662
Trial 2 achieved value: 0.6714289239905273 with  37.7924% improvement
Trial 8 achieved value: 0.6717685767221535 with  0.0506% improvement
Trial 11 achieved value: 0.6750547660777 with  0.4868% improvement
Trial 12 achieved value: 0.6761756373519199 with  0.1658% improvement
Trial 13 achieved value: 0.6793896253614197 with  0.4731% improvement
Trial 15 achieved value: 0.6797632502611368 with  0.0550% improvement
Trial 17 achieved value: 0.6806506050041383 with  0.1304% improvement
Trial 21 achieved value: 0.680693062170169 with  0.0062% improvement
Trial 23 achieved value: 0.6807737303980692 with  0.0118% improvement
Trial 25 achieved value: 0.6810327150820548 with  0.0380% improvement


/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and 

Trial 41 achieved value: 0.6810879093888818 with  0.0081% improvement


/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and 

...

Trial 136 achieved value: 0.7022357955150322 with  1.0955% improvement


/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and 

Trial 152 achieved value: 0.7105403992367735 with  1.1688% improvement


/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/thananpornsethjinda/Library/Caches/pypoetry/virtualenvs/credit-risk-RG99JJ23-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and 

Optimisation works!


In [ ]:
os.makedirs("../src/model/optuna", exist_ok=True)
joblib.dump(study, "../src/model/optuna/final_model_study.pkl")

['../src/model/optuna/final_model_study.pkl']

In [25]:
# best hyperparameters 

study.best_params

{'n_estimators': 177,
 'booster': 'gbtree',
 'learning_rate': 0.01005215884865718,
 'max_depth': 1,
 'min_child_weight': 3,
 'objective': 'binary:logistic'}

In [49]:
best_params = {
    'random_state': 42, 
    'scale_pos_weight': class_ratio,
    'n_estimators': 177,
    'booster': 'gbtree',
    'learning_rate': 0.01005215884865718,
    'max_depth': 1,
    'min_child_weight': 3,
    'objective': 'binary:logistic'
}

### Analysis of Hyperparameter Tuning Results

In [4]:
study = joblib.load("/Users/thananpornsethjinda/Desktop/credit-risk-modeling/src/model/optuna/final_model_study.pkl")

# Now you can plot without rerunning the optimization
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [5]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()